In [ ]:
import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    Dense,
    Dropout,
    Reshape,
    Multiply,
    GlobalAveragePooling2D,
    MultiHeadAttention,
    LayerNormalization,
    GlobalAveragePooling1D
)

from tensorflow.keras.models import Model

In [ ]:
input_layer = Input(
    shape=(64, 64, 32)
)

x = Conv2D(
    32,
    (3,3),
    activation='relu',
    padding='same'
)(input_layer)

attention_map = Conv2D(
    32,
    (1,1),
    activation='sigmoid'
)(x)

x = Multiply()([
    x,
    attention_map
])

x = MaxPooling2D((2,2))(x)

x = Conv2D(
    64,
    (3,3),
    activation='relu',
    padding='same'
)(x)

x = MaxPooling2D((2,2))(x)

x = Reshape(
    (16*16, 64)
)(x)

attention_output = MultiHeadAttention(
    num_heads=4,
    key_dim=64
)(
    x,
    x
)

x = LayerNormalization()(attention_output)

x = GlobalAveragePooling1D()(x)

x = Dropout(0.5)(x)

x = Dense(
    64,
    activation='relu'
)(x)

output_layer = Dense(
    2,
    activation='softmax'
)(x)

amdet_model = Model(
    inputs=input_layer,
    outputs=output_layer
)

amdet_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

amdet_model.summary()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights = dict(
    enumerate(class_weights)
)

print(class_weights)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(
        X_test,
        y_test
    ),
    epochs=15,
    batch_size=8,
    class_weight=class_weights
)

In [ ]:
plt.plot(
    history_amdet.history['accuracy']
)

plt.plot(
    history_amdet.history['val_accuracy']
)

plt.title(
    "AMDET Model Accuracy"
)

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.legend([
    'Train',
    'Validation'
])

plt.savefig(
    "/kaggle/working/amdet_accuracy.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
plt.plot(
    history_amdet.history['loss']
)

plt.plot(
    history_amdet.history['val_loss']
)

plt.title(
    "AMDET Model Loss"
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.legend([
    'Train',
    'Validation'
])

plt.savefig(
    "/kaggle/working/amdet_loss.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
sample = X_test[0]

prediction = amdet_model.predict(
    sample[np.newaxis, ...]
)

predicted_class = np.argmax(
    prediction
)

print(prediction)

print(predicted_class)

In [ ]:
plt.figure(figsize=(8,8))

plt.imshow(
    sample[:, :, 0],
    cmap='inferno'
)

plt.title(
    f"AMDET Spectrogram | Predicted: {predicted_class}"
)

plt.colorbar()

plt.savefig(
    "/kaggle/working/amdet_spectrogram.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
attention_model = Model(
    inputs=amdet_model.input,
    outputs=amdet_model.layers[8].output
)

attention_features = attention_model.predict(
    sample[np.newaxis, ...]
)

attention_map = np.mean(
    attention_features[0],
    axis=-1
)

plt.figure(figsize=(10,4))

plt.plot(attention_map)

plt.title(
    "AMDET Attention Importance"
)

plt.xlabel("Token")

plt.ylabel("Importance")

plt.savefig(
    "/kaggle/working/amdet_attention.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
grad_model = tf.keras.models.Model(
    [amdet_model.inputs],
    [
        amdet_model.get_layer(index=3).output,
        amdet_model.output
    ]
)

input_image = sample[np.newaxis, ...]

with tf.GradientTape() as tape:

    conv_outputs, predictions = grad_model(
        input_image
    )

    class_idx = tf.argmax(
        predictions[0]
    )

    loss = predictions[:, class_idx]

grads = tape.gradient(
    loss,
    conv_outputs
)

pooled_grads = tf.reduce_mean(
    grads,
    axis=(0,1,2)
)

conv_outputs = conv_outputs[0]

heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]

heatmap = tf.squeeze(heatmap)

heatmap = np.maximum(
    heatmap,
    0
)

heatmap /= np.max(
    heatmap
)

plt.figure(figsize=(8,8))

plt.imshow(
    sample[:, :, 0],
    cmap='gray'
)

plt.imshow(
    heatmap,
    cmap='jet',
    alpha=0.5
)

plt.title(
    "AMDET GradCAM"
)

plt.colorbar()

plt.savefig(
    "/kaggle/working/amdet_gradcam.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
real_importance = np.std(
    sample,
    axis=(0,1)
)

real_importance = (
    real_importance -
    np.min(real_importance)
)

real_importance = (
    real_importance /
    np.max(real_importance)
)

fig, ax = plt.subplots(figsize=(8,8))

mne.viz.plot_topomap(
    real_importance,
    info,
    cmap='jet',
    contours=6,
    axes=ax,
    show=False
)

plt.title(
    "AMDET EEG Topomap"
)

plt.savefig(
    "/kaggle/working/amdet_topomap.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
predictions = amdet_model.predict(
    X_test
)

y_pred = np.argmax(
    predictions,
    axis=1
)

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

acc = accuracy_score(
    y_test,
    y_pred
)

report = classification_report(
    y_test,
    y_pred
)

cm = confusion_matrix(
    y_test,
    y_pred
)

with open(
    "/kaggle/working/amdet_results.txt",
    "w"
) as f:

    f.write(f"Accuracy: {acc}\n\n")

    f.write(report)

print(report)

In [ ]:
import seaborn as sns

plt.figure(figsize=(6,6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title(
    "AMDET Confusion Matrix"
)

plt.xlabel("Predicted")

plt.ylabel("True")

plt.savefig(
    "/kaggle/working/amdet_confusion_matrix.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()